In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("Tensors created successfully.")

In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# 4. Print shape of one batch
images_batch, labels_batch = next(iter(train_loader))
print(f"Batch images shape::{images_batch.shape}")
print(f"Batch labels shape:{labels_batch.shape}")

In [ ]:
# 5. Display sample images
plt.figure(figsize=(12,4))
for i in range(5):
    plt.subplot(1,5,i+1)

    display_img = images_batch[i].permute(1,2 ,0).numpy()
    plt.imshow(display_img)
    plt.title(f"Age: {int(labels_batch[i].item())}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
import torch.optim as optim

class AgePredictionModel(nn.Module):
    def __init__(self, input_size):
        super(AgePredictionModel, self).__init__()
        # Layer 1:
        self.fc1 = nn.Linear(input_size, 512)
        # Layer2:
        self.fc2 = nn.Linear(512, 256)
        # Layer3:
        self.fc3 = nn.Linear(256, 64)
        # Layer4:
        self.fc4 = nn.Linear(64, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
# Task 2: Write your training loop here:

def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, ages in train_loader:
        images, ages = images.to(device), ages.to(device)

        outputs = model(images)
        loss = criterion(outputs, ages)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(train_loader)

In [ ]:
# Task 3: Write your validation loop here:

def validate_model(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, ages in test_loader:
            images, ages = images.to(device), ages.to(device)
            outputs = model(images)
            loss = criterion(outputs, ages)
            total_loss += loss.item()
    return total_loss / len(test_loader)


In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = X_train.shape[1] * X_train.shape[2] * X_train.shape[3]

model = AgePredictionModel(input_size=input_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []

print("Starting train")
for epoch in range(20):
    train_loss = train_model(model, train_loader, criterion, optimizer, device)
    val_loss = validate_model(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"epochs: {epoch+1}/20: Train Loss= { train_loss:.4f}, Val Loss= {val_loss:.4f}")

print("Train completes")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='red')

plt.title('Model Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
model.eval()
images, ages = next(iter(test_loader))
images, ages = images.to(device), ages.to(device)

with torch.no_grad():
    predictions = model(images)

plt.figure(figsize=(15,5))
for i in range(5):
    plt.subplot(1,5 , i+1)
    img = images[i].cpu().permute(1 ,2 ,0).numpy()
    plt.imshow(img)

    actual_age = ages[i].item()
    pred_age = predictions[i].item()
    plt.title(f"actual: {actual_age:.0f}\n\n:{pred_age:.1f}")
    plt.axis('off')

plt.tight_layout()
plt.show()